# Flowmetric — SQL Analysis
### Portfolio Project | SQLite + Python

This notebook demonstrates advanced SQL techniques applied to the Flowmetric  
B2B SaaS dataset. Data is pre-cleaned (transformations documented in  
`flowmetric_eda.ipynb`) and loaded into SQLite before analysis.

**Database:** SQLite (persisted to `data/flowmetric_clean.db`)  
**Tables:** customers, plans, subscriptions, events, payments

## Techniques Demonstrated
- Common Table Expressions (CTEs)
- Window Functions: `SUM OVER`, `RANK OVER`, `LAG OVER`, `NTILE`
- Date arithmetic with `JULIANDAY()` and `STRFTIME()`
- Multi-table JOINs (up to 4 tables)
- Conditional aggregation with `CASE WHEN` inside `SUM()`
- `UNION ALL` for inline reference tables

## Table of Contents
1. [Setup & Data Load](#1-setup--data-load)
2. [Block 1 — Revenue Health](#2-block-1--revenue-health)
3. [Block 2 — Churn Analysis](#3-block-2--churn-analysis)
4. [Block 3 — Customer LTV](#4-block-3--customer-ltv)
5. [Block 4 — Engagement & Early Warning](#5-block-4--engagement--early-warning)

In [1]:
import sqlite3
import pandas as pd

# ── Load and clean data before ingestion into SQLite ──────────────────────
customers     = pd.read_csv("../data/raw/customers.csv")
plans         = pd.read_csv("../data/raw/plans.csv")
subscriptions = pd.read_csv("../data/raw/subscriptions.csv")
events        = pd.read_csv("../data/raw/events.csv")
payments      = pd.read_csv("../data/raw/payments.csv")

# Clean customers
customers["country"] = (
    customers["country"]
    .str.strip()
    .str.title()
    .str.replace(r"^Untd\s+", "United ", regex=True)
)
customers["industry"] = customers["industry"].fillna("Unknown")

# Clean events
events = events.drop_duplicates()
events["event_type"] = events["event_type"].fillna("unknown")

# ── Load cleaned data into SQLite ─────────────────────────────────────────
conn = sqlite3.connect("../data/flowmetric_clean.db")

for name, df in [
    ("customers",     customers),
    ("plans",         plans),
    ("subscriptions", subscriptions),
    ("events",        events),
    ("payments",      payments),
]:
    df.to_sql(name, conn, if_exists="replace", index=False)
    print(f"Loaded {name:>15} — {len(df):>7,} rows")

print("\nClean database ready ✓")

Loaded       customers —   1,000 rows
Loaded           plans —       3 rows
Loaded   subscriptions —   1,094 rows
Loaded          events — 200,322 rows
Loaded        payments —  11,410 rows

Clean database ready ✓


## 1. Setup & Data Load

Data loaded from CSV files with cleaning transformations applied before  
ingestion — simulating a production ETL pipeline where raw data is cleaned  
before reaching the analytical database.

**Cleaning applied:**
- `customers.country` — normalised casing, stripped whitespace, fixed abbreviations
- `customers.industry` — 20 missing values filled as `'Unknown'`
- `events` — 1,983 duplicate rows removed before load
- `events.event_type` — 1,012 missing values filled as `'unknown'`

| Table | Rows loaded |
|---|---|
| customers | 1,000 |
| plans | 3 |
| subscriptions | 1,094 |
| events | 200,322 |
| payments | 11,410 |

In [2]:
# Helper function — execute SQL and return DataFrame
def run_query(sql, conn=conn):
    return pd.read_sql_query(sql, conn)

# Verify tables
print("Tables in database:")
for table in ["customers", "plans", "subscriptions", "events", "payments"]:
    count = run_query(f"SELECT COUNT(*) as n FROM {table}").iloc[0]["n"]
    print(f"  {table:>15}: {count:>7,} rows")

print("\nDatabase ready ✓")

Tables in database:
        customers:   1,000 rows
            plans:       3 rows
    subscriptions:   1,094 rows
           events: 200,322 rows
         payments:  11,410 rows

Database ready ✓


## 2. Block 1 — Revenue Health

**Techniques demonstrated:** CTEs, Window Functions (`SUM OVER`),
`CASE WHEN`, `SUBSTR` for date extraction

### KPI 1 — Monthly MRR Trend

In [3]:
# KPI 1: Monthly MRR with cumulative total using Window Function
run_query("""
WITH months(month) AS (
    SELECT '2023-07' UNION ALL SELECT '2023-08' UNION ALL SELECT '2023-09'
    UNION ALL SELECT '2023-10' UNION ALL SELECT '2023-11' UNION ALL SELECT '2023-12'
    UNION ALL SELECT '2024-01' UNION ALL SELECT '2024-02' UNION ALL SELECT '2024-03'
    UNION ALL SELECT '2024-04' UNION ALL SELECT '2024-05' UNION ALL SELECT '2024-06'
    UNION ALL SELECT '2024-07' UNION ALL SELECT '2024-08' UNION ALL SELECT '2024-09'
    UNION ALL SELECT '2024-10' UNION ALL SELECT '2024-11' UNION ALL SELECT '2024-12'
    UNION ALL SELECT '2025-01' UNION ALL SELECT '2025-02' UNION ALL SELECT '2025-03'
    UNION ALL SELECT '2025-04' UNION ALL SELECT '2025-05' UNION ALL SELECT '2025-06'
    UNION ALL SELECT '2025-07' UNION ALL SELECT '2025-08' UNION ALL SELECT '2025-09'
    UNION ALL SELECT '2025-10' UNION ALL SELECT '2025-11' UNION ALL SELECT '2025-12'
    UNION ALL SELECT '2026-01' UNION ALL SELECT '2026-02' UNION ALL SELECT '2026-03'
    UNION ALL SELECT '2026-04' UNION ALL SELECT '2026-05' UNION ALL SELECT '2026-06'
),
active_subs AS (
    SELECT
        m.month,
        SUM(s.mrr_amount)             AS total_mrr,
        COUNT(DISTINCT s.customer_id) AS n_customers
    FROM months m
    JOIN subscriptions s
        ON SUBSTR(s.start_date, 1, 7) <= m.month
        AND (
            s.end_date IS NULL
            OR SUBSTR(s.end_date, 1, 7) >= m.month
        )
    GROUP BY m.month
)
SELECT
    month,
    total_mrr,
    n_customers,
    SUM(total_mrr) OVER (ORDER BY month) AS cumulative_mrr
FROM active_subs
ORDER BY month
""")

,month,total_mrr,n_customers,cumulative_mrr
0,2023-07,3689,11,3689
1,2023-08,6429,21,10118
2,2023-09,7871,29,17989
3,2023-10,13256,44,31245
4,2023-11,15148,52,46393
5,2023-12,19881,69,66274
6,2024-01,21574,76,87848
7,2024-02,28502,97,116350
8,2024-03,34386,113,150736
9,2024-04,38363,137,189099


### KPI 1 — Monthly MRR Trend

**SQL techniques:** `UNION ALL` to build inline month spine, `JOIN` with
date range condition, `SUM() OVER (ORDER BY month)` Window Function
for cumulative total

| Metric | Value |
|---|---|
| MRR — July 2023 | €3,689 |
| MRR — June 2026 | €222,317 |
| Total cumulative revenue (36 months) | €3,528,250 |
| Growth | +5,927% |

**Key technique:**  
The `months` CTE uses `UNION ALL` to build a month spine — a common pattern  
in SQLite which lacks `GENERATE_SERIES()`. The `JOIN` condition filters  
subscriptions active in each month using string date comparison.  
`SUM() OVER (ORDER BY month)` computes the running total without a self-join.

### KPI 2 — MRR Movement Breakdown (New / Expansion / Contraction / Churned)

**SQL techniques:** CTE, `CASE WHEN` for movement classification,
`GROUP BY`, `SUM`, `ROUND`

In [4]:
# KPI 2: MRR Movement Breakdown
run_query("""
WITH mrr_movement AS (
    SELECT
        CASE
            WHEN status = 'active'     THEN SUBSTR(start_date, 1, 7)
            WHEN status = 'churned'    THEN SUBSTR(end_date, 1, 7)
            WHEN status = 'upgraded'   THEN SUBSTR(end_date, 1, 7)
            WHEN status = 'downgraded' THEN SUBSTR(end_date, 1, 7)
        END                           AS month,
        CASE
            WHEN status = 'active'     THEN 'New'
            WHEN status = 'churned'    THEN 'Churned'
            WHEN status = 'upgraded'   THEN 'Expansion'
            WHEN status = 'downgraded' THEN 'Contraction'
        END                           AS movement_type,
        mrr_amount
    FROM subscriptions
)
SELECT
    movement_type,
    COUNT(*)                         AS n_events,
    SUM(mrr_amount)                  AS total_mrr,
    ROUND(AVG(mrr_amount), 0)        AS avg_mrr
FROM mrr_movement
WHERE month BETWEEN '2023-07' AND '2026-06'
GROUP BY movement_type
ORDER BY total_mrr DESC
""")

,movement_type,n_events,total_mrr,avg_mrr
0,New,654,216546,331.0
1,Churned,346,48154,139.0
2,Contraction,46,18754,408.0
3,Expansion,48,7002,146.0


### KPI 2 — MRR Movement Breakdown

**SQL techniques:** CTE, `CASE WHEN` for movement type classification,
`GROUP BY`, `SUM`, `AVG`, `ROUND`

| Movement Type | Events | Total MRR | Avg MRR |
|---|---|---|---|
| New | 654 | €216,546 | €331 |
| Churned | 346 | €48,154 | €139 |
| Contraction | 46 | €18,754 | €408 |
| Expansion | 48 | €7,002 | €146 |

**Key technique:**  
A single `CASE WHEN` inside the CTE classifies each subscription row  
into a movement type based on its status. This avoids multiple queries  
or UNION statements — one pass through the table produces all four categories.

**Key insight:**  
Contraction MRR (€18,754) exceeds Expansion MRR (€7,002) — customers  
are downgrading more than upgrading. The average contraction event (€408)  
is also larger than the average expansion event (€146), meaning downgrades  
tend to be from higher-tier plans.

### KPI 3 — Net Revenue Retention (NRR)

**SQL techniques:** `run_query()` for data extraction via SQL + Python for
12-month cohort matching aggregation.

**Note on approach:**  
NRR requires tracking the same customer cohort across two time points.  
In SQLite this is difficult without a calendar table — so we extract the  
raw subscription data via SQL and perform the 12-month window calculation  
in Python. In a production environment (PostgreSQL, BigQuery) this would  
be a pure SQL query using `GENERATE_SERIES()` or a date dimension table.  
This hybrid approach demonstrates knowing **when to combine tools**  
rather than forcing a pure SQL solution that would be harder to read and maintain.

In [5]:
# KPI 3: NRR — SQL extraction + Python cohort calculation
mrr_by_customer = run_query("""
    SELECT
        customer_id,
        SUBSTR(start_date, 1, 7)                AS start_month,
        COALESCE(
            SUBSTR(end_date, 1, 7),
            '2026-06'
        )                                        AS end_month,
        mrr_amount                               AS mrr
    FROM subscriptions
    WHERE status IN ('active', 'churned')
""")

months = pd.date_range(
    "2024-07-01", "2026-06-01", freq="MS"
).strftime("%Y-%m")

nrr_records = []
for m in months:
    m_prev = (pd.Timestamp(m) - pd.DateOffset(months=12)).strftime("%Y-%m")

    cohort = mrr_by_customer[
        (mrr_by_customer["start_month"] <= m_prev) &
        (mrr_by_customer["end_month"]   >= m_prev)
    ]["customer_id"].unique()

    mrr_then = mrr_by_customer[
        (mrr_by_customer["customer_id"].isin(cohort)) &
        (mrr_by_customer["start_month"] <= m_prev) &
        (mrr_by_customer["end_month"]   >= m_prev)
    ]["mrr"].sum()

    mrr_now = mrr_by_customer[
        (mrr_by_customer["customer_id"].isin(cohort)) &
        (mrr_by_customer["start_month"] <= m) &
        (mrr_by_customer["end_month"]   >= m)
    ]["mrr"].sum()

    nrr = round(100 * mrr_now / mrr_then, 1) if mrr_then > 0 else None
    nrr_records.append({
        "month"   : m,
        "mrr_then": mrr_then,
        "mrr_now" : mrr_now,
        "nrr_pct" : nrr,
    })

nrr_df = pd.DataFrame(nrr_records)
print(f"Average NRR: {nrr_df['nrr_pct'].mean():.1f}%")
print(f"Min NRR:     {nrr_df['nrr_pct'].min():.1f}%")
print(f"Max NRR:     {nrr_df['nrr_pct'].max():.1f}%")
nrr_df

Average NRR: 82.0%
Min NRR:     67.6%
Max NRR:     91.5%


,month,mrr_then,mrr_now,nrr_pct
0,2024-07,2492,1846,74.1
1,2024-08,5232,3539,67.6
2,2024-09,6426,4733,73.7
3,2024-10,10164,7277,71.6
4,2024-11,11658,8722,74.8
5,2024-12,15393,12959,84.2
6,2025-01,16088,14404,89.5
7,2025-02,22122,20242,91.5
8,2025-03,27657,24583,88.9
9,2025-04,30835,25878,83.9


### KPI 3 — Net Revenue Retention (NRR)

| Metric | Value |
|---|---|
| Average NRR (months 13–36) | 82.0% |
| Min NRR | 67.6% (2024-08 — early cohorts, small base) |
| Max NRR | 91.5% (2025-02) |
| Industry benchmark | > 100% |

**Interpretation:**  
NRR of 82.0% means Flowmetric retains €82 of every €100 from existing  
customers over 12 months. Below the 100% industry benchmark, indicating  
churn and contraction are outpacing expansion revenue.  
The business is growing only because new customer acquisition is strong —  
a risk if acquisition slows down.

## 3. Block 2 — Churn Analysis

**Techniques demonstrated:** `JOIN`, `CASE WHEN` inside `SUM()` for
conditional counting, `JULIANDAY()` for date differences, `GROUP BY`, `ROUND`

### KPI 4 — Churn Rate by Plan

In [6]:
# KPI 4: Churn rate by plan
run_query("""
SELECT
    p.plan_name,
    COUNT(*)                                               AS total_customers,
    SUM(CASE WHEN s.status = 'churned' THEN 1 ELSE 0 END) AS churned,
    ROUND(
        100.0 * SUM(CASE WHEN s.status = 'churned' THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                      AS churn_rate_pct
FROM subscriptions s
JOIN plans p ON s.plan_id = p.plan_id
WHERE s.status IN ('active', 'churned')
GROUP BY p.plan_name
ORDER BY churn_rate_pct DESC
""")

,plan_name,total_customers,churned,churn_rate_pct
0,Starter,434,226,52.1
1,Pro,348,98,28.2
2,Enterprise,218,22,10.1


### KPI 4 — Churn Rate by Plan

**SQL techniques:** `JOIN` between subscriptions and plans,
`SUM(CASE WHEN status = 'churned' THEN 1 ELSE 0 END)` for conditional
counting, `ROUND`, `GROUP BY`, `ORDER BY`

| Plan | Customers | Churned | Churn Rate |
|---|---|---|---|
| Starter | 434 | 226 | 52.1% |
| Pro | 348 | 98 | 28.2% |
| Enterprise | 218 | 22 | 10.1% |

**Key technique:**  
`SUM(CASE WHEN status = 'churned' THEN 1 ELSE 0 END)` is the standard  
SQL pattern for conditional counting — equivalent to `COUNT(*)` with a  
`WHERE` clause but allows multiple conditions in a single `GROUP BY` query.

**Key insight:**  
Starter churn rate (52.1%) is 5x higher than Enterprise (10.1%),  
confirming that self-serve low-cost plans have significantly higher  
churn than annual enterprise contracts.

### KPI 5 — Churn Rate by Country and Industry

**SQL techniques:** multi-table JOIN (subscriptions + customers),
`CASE WHEN` inside `SUM()`, `WHERE` with `IN` clause, `GROUP BY`, `ORDER BY`

In [7]:
# KPI 5a: Churn rate by country
run_query("""
SELECT
    c.country,
    COUNT(*)                                               AS total_customers,
    SUM(CASE WHEN s.status = 'churned' THEN 1 ELSE 0 END) AS churned,
    ROUND(
        100.0 * SUM(CASE WHEN s.status = 'churned' THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                      AS churn_rate_pct
FROM subscriptions s
JOIN customers c ON s.customer_id = c.customer_id
WHERE s.status IN ('active', 'churned')
AND c.country IN (
    'United States', 'United Kingdom', 'Germany', 'Portugal'
)
GROUP BY c.country
ORDER BY churn_rate_pct DESC
""")

,country,total_customers,churned,churn_rate_pct
0,United Kingdom,272,98,36.0
1,United States,387,134,34.6
2,Germany,199,68,34.2
3,Portugal,142,46,32.4


In [8]:
# KPI 5b: Churn rate by industry
run_query("""
SELECT
    c.industry,
    COUNT(*)                                               AS total_customers,
    SUM(CASE WHEN s.status = 'churned' THEN 1 ELSE 0 END) AS churned,
    ROUND(
        100.0 * SUM(CASE WHEN s.status = 'churned' THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                      AS churn_rate_pct
FROM subscriptions s
JOIN customers c ON s.customer_id = c.customer_id
WHERE s.status IN ('active', 'churned')
AND c.industry != 'Unknown'
GROUP BY c.industry
ORDER BY churn_rate_pct DESC
""")

,industry,total_customers,churned,churn_rate_pct
0,Dev Shop / Software Consulting,148,56,37.8
1,Digital Agency,141,52,36.9
2,Marketing Agency,140,48,34.3
3,Management Consulting,137,46,33.6
4,PR Agency,150,50,33.3
5,Design Studio,130,43,33.1
6,Branding Agency,134,42,31.3


### KPI 5 — Churn Rate by Country and Industry

**SQL techniques:** JOIN (subscriptions + customers), `CASE WHEN` inside
`SUM()`, `WHERE IN` clause, `GROUP BY`, `ORDER BY`

**By Country:**

| Country | Customers | Churned | Churn Rate |
|---|---|---|---|
| United Kingdom | 272 | 98 | 36.0% |
| United States | 387 | 134 | 34.6% |
| Germany | 199 | 68 | 34.2% |
| Portugal | 142 | 46 | 32.4% |

**By Industry:**

| Industry | Customers | Churned | Churn Rate |
|---|---|---|---|
| Dev Shop / Software Consulting | 148 | 56 | 37.8% |
| Digital Agency | 141 | 52 | 36.9% |
| Marketing Agency | 140 | 48 | 34.3% |
| Management Consulting | 137 | 46 | 33.6% |
| PR Agency | 150 | 50 | 33.3% |
| Design Studio | 130 | 43 | 33.1% |
| Branding Agency | 134 | 42 | 31.3% |

**Key insight:**  
Geographic and industry churn differences are small (31–38%) compared  
to plan-tier differences (10–52%). Churn is primarily a product and  
onboarding problem, not a market or segment problem.

### KPI 6 — Time-to-Churn by Plan

**SQL techniques:** `JULIANDAY()` for date difference calculation in SQLite,
`AVG / MIN / MAX` with `CASE WHEN`, `JOIN`, `GROUP BY`

In [9]:
# KPI 6: Time to churn by plan
run_query("""
SELECT
    p.plan_name,
    COUNT(CASE WHEN s.status = 'churned' THEN 1 END) AS churned_customers,
    ROUND(AVG(
        CASE WHEN s.status = 'churned'
        THEN JULIANDAY(s.end_date) - JULIANDAY(s.start_date)
        END
    ), 0)                                             AS avg_days_to_churn,
    ROUND(MIN(
        CASE WHEN s.status = 'churned'
        THEN JULIANDAY(s.end_date) - JULIANDAY(s.start_date)
        END
    ), 0)                                             AS min_days,
    ROUND(MAX(
        CASE WHEN s.status = 'churned'
        THEN JULIANDAY(s.end_date) - JULIANDAY(s.start_date)
        END
    ), 0)                                             AS max_days
FROM subscriptions s
JOIN plans p ON s.plan_id = p.plan_id
WHERE s.status IN ('active', 'churned')
GROUP BY p.plan_name
ORDER BY avg_days_to_churn DESC
""")

,plan_name,churned_customers,avg_days_to_churn,min_days,max_days
0,Enterprise,22,274.0,-9.0,584.0
1,Pro,98,216.0,-26.0,625.0
2,Starter,226,201.0,-27.0,1000.0


### KPI 6 — Time-to-Churn by Plan

**SQL techniques:** `JULIANDAY()` for date difference in SQLite,
`AVG / MIN / MAX` with `CASE WHEN`, `JOIN`, `GROUP BY`

| Plan | Churned | Avg Days to Churn | Min | Max |
|---|---|---|---|---|
| Enterprise | 22 | 274 days | -9 | 584 |
| Pro | 98 | 216 days | -26 | 625 |
| Starter | 226 | 201 days | -27 | 1,000 |

**Note on negative values:**  
Minimum days shows negative values in the raw subscription dates — these  
are the 33 records identified and corrected in the EDA notebook where  
`start_date` falls after `end_date` due to day-of-month jitter in the  
data generator. The `avg_days_to_churn` values are not materially affected  
(33 records out of 346 churned). In a production pipeline the fix would  
be applied at source before database ingestion.

**Key insight:**  
Enterprise customers survive 35% longer before churning (274 vs 201 days),  
reflecting longer contract cycles and higher switching costs.

## 4. Block 3 — Customer Lifetime Value (LTV)

**Techniques demonstrated:** CTEs, multi-table JOIN (up to 4 tables),
`RANK() OVER()` Window Function, `UNION ALL` for inline reference tables,
`SUM`, `AVG`, `MIN`, `MAX`

### KPI 7 — LTV by Plan

In [10]:
# KPI 7: Average LTV by plan
run_query("""
WITH customer_revenue AS (
    SELECT
        customer_id,
        SUM(amount) AS total_revenue
    FROM payments
    WHERE status = 'succeeded'
    GROUP BY customer_id
)
SELECT
    p.plan_name,
    COUNT(DISTINCT s.customer_id)        AS total_customers,
    ROUND(AVG(cr.total_revenue), 0)      AS avg_ltv,
    ROUND(MIN(cr.total_revenue), 0)      AS min_ltv,
    ROUND(MAX(cr.total_revenue), 0)      AS max_ltv,
    ROUND(SUM(cr.total_revenue), 0)      AS total_revenue
FROM subscriptions s
JOIN plans p             ON s.plan_id = p.plan_id
JOIN customer_revenue cr ON s.customer_id = cr.customer_id
WHERE s.status IN ('active', 'churned')
GROUP BY p.plan_name
ORDER BY avg_ltv DESC
""")

,plan_name,total_customers,avg_ltv,min_ltv,max_ltv,total_revenue
0,Enterprise,216,10769.0,799.0,28764.0,2326049.0
1,Pro,342,2459.0,199.0,24567.0,840933.0
2,Starter,407,659.0,49.0,18423.0,268157.0


### KPI 7 — Customer LTV by Plan

**SQL techniques:** CTE (`customer_revenue`), 3-table JOIN,
`SUM / AVG / MIN / MAX`, `GROUP BY`, `ORDER BY`

| Plan | Customers | Avg LTV | Min LTV | Max LTV | Total Revenue |
|---|---|---|---|---|---|
| Enterprise | 216 | €10,769 | €799 | €28,764 | €2,326,049 |
| Pro | 342 | €2,459 | €199 | €24,567 | €840,933 |
| Starter | 407 | €659 | €49 | €18,423 | €268,157 |

**Key technique:**  
The `customer_revenue` CTE pre-aggregates all succeeded payments per customer  
before joining with subscriptions and plans. This is more efficient than a  
correlated subquery and makes the logic easier to read and debug — the CTE  
acts as a reusable building block for subsequent queries in this block.

**Key insight:**  
Enterprise LTV (€10,769) is 16x higher than Starter (€659).  
Despite being only 22% of customers, Enterprise generates 68% of total revenue.

### KPI 8 — LTV by Country and Industry

**SQL techniques:** CTE, 3-table JOIN, `RANK() OVER (ORDER BY avg_ltv DESC)`
Window Function, `GROUP BY`, `ROUND`

In [11]:
# KPI 8a: LTV by country with RANK window function
run_query("""
WITH customer_revenue AS (
    SELECT
        customer_id,
        SUM(amount) AS total_revenue
    FROM payments
    WHERE status = 'succeeded'
    GROUP BY customer_id
),
ltv_by_country AS (
    SELECT
        c.country,
        COUNT(DISTINCT s.customer_id)    AS total_customers,
        ROUND(AVG(cr.total_revenue), 0)  AS avg_ltv,
        ROUND(SUM(cr.total_revenue), 0)  AS total_revenue
    FROM subscriptions s
    JOIN customers c         ON s.customer_id = c.customer_id
    JOIN customer_revenue cr ON s.customer_id = cr.customer_id
    WHERE s.status IN ('active', 'churned')
    AND c.country IN (
        'United States', 'United Kingdom', 'Germany', 'Portugal'
    )
    GROUP BY c.country
)
SELECT
    country,
    total_customers,
    avg_ltv,
    total_revenue,
    RANK() OVER (ORDER BY avg_ltv DESC) AS ltv_rank
FROM ltv_by_country
ORDER BY ltv_rank
""")

,country,total_customers,avg_ltv,total_revenue,ltv_rank
0,United States,373,3846.0,1434554.0,1
1,Portugal,139,3656.0,508178.0,2
2,Germany,188,3498.0,657544.0,3
3,United Kingdom,265,3150.0,834863.0,4


In [12]:
# KPI 8b: LTV by industry with RANK window function
run_query("""
WITH customer_revenue AS (
    SELECT
        customer_id,
        SUM(amount) AS total_revenue
    FROM payments
    WHERE status = 'succeeded'
    GROUP BY customer_id
),
ltv_by_industry AS (
    SELECT
        c.industry,
        COUNT(DISTINCT s.customer_id)    AS total_customers,
        ROUND(AVG(cr.total_revenue), 0)  AS avg_ltv,
        ROUND(SUM(cr.total_revenue), 0)  AS total_revenue
    FROM subscriptions s
    JOIN customers c         ON s.customer_id = c.customer_id
    JOIN customer_revenue cr ON s.customer_id = cr.customer_id
    WHERE s.status IN ('active', 'churned')
    AND c.industry != 'Unknown'
    GROUP BY c.industry
)
SELECT
    industry,
    total_customers,
    avg_ltv,
    total_revenue,
    RANK() OVER (ORDER BY avg_ltv DESC) AS ltv_rank
FROM ltv_by_industry
ORDER BY ltv_rank
""")

,industry,total_customers,avg_ltv,total_revenue,ltv_rank
0,Dev Shop / Software Consulting,144,4306.0,620079.0,1
1,Management Consulting,133,3923.0,521809.0,2
2,Branding Agency,129,3841.0,495463.0,3
3,Marketing Agency,131,3574.0,468129.0,4
4,Design Studio,123,3559.0,437729.0,5
5,Digital Agency,138,3208.0,442635.0,6
6,PR Agency,147,2711.0,398454.0,7


### KPI 8 — LTV by Country and Industry

**SQL techniques:** 2-CTE chain, 3-table JOIN, `RANK() OVER (ORDER BY avg_ltv DESC)`,
`GROUP BY`, `ROUND`

**By Country:**

| Rank | Country | Customers | Avg LTV | Total Revenue |
|---|---|---|---|---|
| 1 | United States | 373 | €3,846 | €1,434,554 |
| 2 | Portugal | 139 | €3,656 | €508,178 |
| 3 | Germany | 188 | €3,498 | €657,544 |
| 4 | United Kingdom | 265 | €3,150 | €834,863 |

**By Industry:**

| Rank | Industry | Customers | Avg LTV | Total Revenue |
|---|---|---|---|---|
| 1 | Dev Shop / Software Consulting | 144 | €4,306 | €620,079 |
| 2 | Management Consulting | 133 | €3,923 | €521,809 |
| 3 | Branding Agency | 129 | €3,841 | €495,463 |
| 4 | Marketing Agency | 131 | €3,574 | €468,129 |
| 5 | Design Studio | 123 | €3,559 | €437,729 |
| 6 | Digital Agency | 138 | €3,208 | €442,635 |
| 7 | PR Agency | 147 | €2,711 | €398,454 |

**Key technique:**  
`RANK() OVER (ORDER BY avg_ltv DESC)` ranks each segment without collapsing  
the result — a cleaner alternative to `ORDER BY` alone when you need the  
rank number as a column for downstream filtering or reporting.

### KPI 9 — LTV / CAC Ratio

**SQL techniques:** 3 chained CTEs, `UNION ALL` to build inline reference
table, `JOIN` between CTEs, `CASE WHEN` for assessment label, `ROUND`

In [13]:
# KPI 9: LTV / CAC Ratio by plan
run_query("""
WITH customer_revenue AS (
    SELECT
        customer_id,
        SUM(amount) AS total_revenue
    FROM payments
    WHERE status = 'succeeded'
    GROUP BY customer_id
),
ltv_by_plan AS (
    SELECT
        p.plan_name,
        ROUND(AVG(cr.total_revenue), 0) AS avg_ltv
    FROM subscriptions s
    JOIN plans p             ON s.plan_id = p.plan_id
    JOIN customer_revenue cr ON s.customer_id = cr.customer_id
    WHERE s.status IN ('active', 'churned')
    GROUP BY p.plan_name
),
cac_assumptions AS (
    SELECT 'Starter'    AS plan_name, 150  AS assumed_cac UNION ALL
    SELECT 'Pro'        AS plan_name, 800  AS assumed_cac UNION ALL
    SELECT 'Enterprise' AS plan_name, 3500 AS assumed_cac
)
SELECT
    l.plan_name,
    l.avg_ltv,
    c.assumed_cac,
    ROUND(1.0 * l.avg_ltv / c.assumed_cac, 2) AS ltv_cac_ratio,
    CASE
        WHEN 1.0 * l.avg_ltv / c.assumed_cac >= 3 THEN 'Healthy (>3x)'
        ELSE 'At risk (<3x)'
    END                                         AS assessment
FROM ltv_by_plan l
JOIN cac_assumptions c ON l.plan_name = c.plan_name
ORDER BY ltv_cac_ratio DESC
""")

,plan_name,avg_ltv,assumed_cac,ltv_cac_ratio,assessment
0,Starter,659.0,150,4.39,Healthy (>3x)
1,Enterprise,10769.0,3500,3.08,Healthy (>3x)
2,Pro,2459.0,800,3.07,Healthy (>3x)


### KPI 9 — LTV / CAC Ratio

**SQL techniques:** 3 chained CTEs (`customer_revenue` → `ltv_by_plan` →
`cac_assumptions`), `UNION ALL` for inline reference table,
`JOIN` between CTEs, `CASE WHEN` for label, `ROUND`

| Plan | Avg LTV | Assumed CAC | LTV/CAC Ratio | Assessment |
|---|---|---|---|---|
| Starter | €659 | €150 | 4.39x | Healthy (>3x) |
| Enterprise | €10,769 | €3,500 | 3.08x | Healthy (>3x) |
| Pro | €2,459 | €800 | 3.07x | Healthy (>3x) |

**Key technique:**  
The `cac_assumptions` CTE uses `UNION ALL` to build an inline reference  
table without requiring a separate lookup table in the database.  
This is a common pattern in SQL analytics for injecting static reference  
values directly into a query — clean, portable and self-documenting.

**Note on CAC:**  
CAC values are modelled estimates based on B2B SaaS industry benchmarks.  
In a production environment this would join to a marketing spend table.

## 5. Block 4 — Engagement & Early Warning

**Techniques demonstrated:** CTE with subquery, `LEFT JOIN` with date
arithmetic, `JULIANDAY()`, `STRFTIME()`, `MAX(CASE WHEN)` boolean flag,
cohort analysis with `GROUP BY`

### KPI 10 — Product Engagement in First 30 Days

In [14]:
# KPI 10: Avg events in first 30 days — active vs churned
run_query("""
WITH customer_signup AS (
    SELECT
        c.customer_id,
        c.signup_date,
        s.status
    FROM customers c
    JOIN subscriptions s ON c.customer_id = s.customer_id
    WHERE s.status IN ('active', 'churned')
),
first30_events AS (
    SELECT
        cs.customer_id,
        cs.status,
        COUNT(e.event_id) AS events_first30
    FROM customer_signup cs
    LEFT JOIN events e
        ON e.customer_id = cs.customer_id
        AND e.event_date >= cs.signup_date
        AND JULIANDAY(e.event_date) - JULIANDAY(cs.signup_date) < 30
    GROUP BY cs.customer_id, cs.status
)
SELECT
    status,
    COUNT(*)                          AS total_customers,
    ROUND(AVG(events_first30), 1)     AS avg_events,
    ROUND(MIN(events_first30), 0)     AS min_events,
    ROUND(MAX(events_first30), 0)     AS max_events
FROM first30_events
GROUP BY status
ORDER BY avg_events DESC
""")

,status,total_customers,avg_events,min_events,max_events
0,active,654,17.7,0.0,38.0
1,churned,346,15.8,0.0,34.0


### KPI 10 — Product Engagement in First 30 Days

**SQL techniques:** 2-CTE chain, `LEFT JOIN` with `JULIANDAY()` date filter,
`COUNT`, `AVG`, `MIN`, `MAX`, `GROUP BY`

| Status | Customers | Avg Events (30d) | Min | Max |
|---|---|---|---|---|
| Active | 654 | 17.7 | 0 | 38 |
| Churned | 346 | 15.8 | 0 | 34 |
| Difference | — | +11.4% | — | — |

**Key technique:**  
The `LEFT JOIN` between `customer_signup` and `events` uses `JULIANDAY()`  
to calculate the number of days between signup and each event date.  
This filters events within the 30-day window without `DATEADD` (unavailable  
in SQLite) — a common SQLite-specific pattern for date range joins.

**Key insight:**  
Customers who churn average 11.4% fewer product events in their first 30 days.  
This statistically significant signal (confirmed in Python analysis, p=0.0002)  
makes early engagement a reliable leading indicator of churn risk.

### KPI 11 — Deep Feature Usage vs Churn Rate (first 90 days)

**SQL techniques:** CTE, `LEFT JOIN` with `JULIANDAY()` date filter,
`MAX(CASE WHEN)` boolean flag pattern, conditional `SUM()`, `GROUP BY`

In [15]:
# KPI 11: Deep feature usage vs churn rate (first 90 days) 
run_query("""
WITH customer_signup AS (
    SELECT
        c.customer_id,
        c.signup_date,
        s.status
    FROM customers c
    JOIN subscriptions s ON c.customer_id = s.customer_id
    WHERE s.status IN ('active', 'churned')
),
deep_usage_90d AS (
    SELECT
        cs.customer_id,
        cs.status,
        COUNT(e.event_id) AS deep_events_90d
    FROM customer_signup cs
    LEFT JOIN events e
        ON  e.customer_id  = cs.customer_id
        AND e.event_type   IN ('time_logged', 'invoice_sent')
        AND e.event_date   >= cs.signup_date
        AND JULIANDAY(e.event_date) - JULIANDAY(cs.signup_date) < 90
    GROUP BY cs.customer_id, cs.status
)
SELECT
    CASE WHEN deep_events_90d > 0
        THEN 'Used deep features'
        ELSE 'Never used deep features'
    END                                                    AS segment,
    COUNT(*)                                               AS total_customers,
    SUM(CASE WHEN status = 'churned' THEN 1 ELSE 0 END)   AS churned,
    ROUND(
        100.0 * SUM(CASE WHEN status = 'churned' THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                      AS churn_rate_pct
FROM deep_usage_90d
GROUP BY segment
ORDER BY churn_rate_pct DESC
""")

,segment,total_customers,churned,churn_rate_pct
0,Never used deep features,25,12,48.0
1,Used deep features,975,334,34.3


### KPI 11 — Deep Feature Usage vs Churn Rate (first 90 days)

**SQL techniques:** CTE, `LEFT JOIN` with multiple conditions including
`JULIANDAY()` date range filter, `COUNT` for event aggregation,
`CASE WHEN` for segment label, conditional `SUM()`, `GROUP BY`

| Segment | Customers | Churned | Churn Rate |
|---|---|---|---|
| Never used deep features | 25 | 12 | 48.0% |
| Used deep features | 975 | 334 | 34.3% |
| Relative risk | — | — | 1.40x more likely to churn |

**Key technique:**  
Moving the date filter into the `LEFT JOIN` condition (rather than inside  
a `CASE WHEN`) ensures only events within the 90-day window are counted.  
The `CASE WHEN deep_events_90d > 0` then cleanly classifies each customer  
as a deep feature user or not — a more readable and correct pattern than  
`MAX(CASE WHEN)` with a filter inside the expression.

**Key insight:**  
Customers who never use `time_logged` or `invoice_sent` in their first  
90 days are 1.40x more likely to churn (48.0% vs 34.3%).  
Deep feature activation is the strongest early warning signal in the dataset.

### KPI 12 — Cohort Retention by Signup Quarter

**SQL techniques:** CTE, `STRFTIME()` for quarter extraction,
`CASE WHEN` for checkpoint columns, `GROUP BY`, `ROUND`

In [16]:
# KPI 12: Cohort retention — fixed with NULLIF for future checkpoints
run_query("""
WITH customer_cohorts AS (
    SELECT
        c.customer_id,
        c.signup_date,
        STRFTIME('%Y', c.signup_date) || '-Q' ||
        CASE
            WHEN STRFTIME('%m', c.signup_date) BETWEEN '01' AND '03' THEN '1'
            WHEN STRFTIME('%m', c.signup_date) BETWEEN '04' AND '06' THEN '2'
            WHEN STRFTIME('%m', c.signup_date) BETWEEN '07' AND '09' THEN '3'
            ELSE '4'
        END                           AS cohort_quarter,
        MIN(c.signup_date)            AS cohort_start
    FROM customers c
    GROUP BY c.customer_id, cohort_quarter
),
cohort_status AS (
    SELECT
        cc.customer_id,
        cc.cohort_quarter,
        cc.signup_date,
        COALESCE(s.end_date, '2026-06-30') AS end_date_filled
    FROM customer_cohorts cc
    JOIN subscriptions s ON cc.customer_id = s.customer_id
    WHERE s.status IN ('active', 'churned')
)
SELECT
    cohort_quarter,
    COUNT(DISTINCT customer_id) AS cohort_size,
    NULLIF(ROUND(100.0 * SUM(CASE WHEN
        JULIANDAY('2026-06-30') - JULIANDAY(signup_date) >= 90
        AND JULIANDAY(end_date_filled) - JULIANDAY(signup_date) >= 90
        THEN 1 ELSE 0 END) / COUNT(DISTINCT customer_id), 1), 0) AS month_3,
    NULLIF(ROUND(100.0 * SUM(CASE WHEN
        JULIANDAY('2026-06-30') - JULIANDAY(signup_date) >= 180
        AND JULIANDAY(end_date_filled) - JULIANDAY(signup_date) >= 180
        THEN 1 ELSE 0 END) / COUNT(DISTINCT customer_id), 1), 0) AS month_6,
    NULLIF(ROUND(100.0 * SUM(CASE WHEN
        JULIANDAY('2026-06-30') - JULIANDAY(signup_date) >= 270
        AND JULIANDAY(end_date_filled) - JULIANDAY(signup_date) >= 270
        THEN 1 ELSE 0 END) / COUNT(DISTINCT customer_id), 1), 0) AS month_9,
    NULLIF(ROUND(100.0 * SUM(CASE WHEN
        JULIANDAY('2026-06-30') - JULIANDAY(signup_date) >= 365
        AND JULIANDAY(end_date_filled) - JULIANDAY(signup_date) >= 365
        THEN 1 ELSE 0 END) / COUNT(DISTINCT customer_id), 1), 0) AS month_12,
    NULLIF(ROUND(100.0 * SUM(CASE WHEN
        JULIANDAY('2026-06-30') - JULIANDAY(signup_date) >= 548
        AND JULIANDAY(end_date_filled) - JULIANDAY(signup_date) >= 548
        THEN 1 ELSE 0 END) / COUNT(DISTINCT customer_id), 1), 0) AS month_18,
    NULLIF(ROUND(100.0 * SUM(CASE WHEN
        JULIANDAY('2026-06-30') - JULIANDAY(signup_date) >= 730
        AND JULIANDAY(end_date_filled) - JULIANDAY(signup_date) >= 730
        THEN 1 ELSE 0 END) / COUNT(DISTINCT customer_id), 1), 0) AS month_24
FROM cohort_status
GROUP BY cohort_quarter
ORDER BY cohort_quarter
""")

,cohort_quarter,cohort_size,month_3,month_6,month_9,month_12,month_18,month_24
0,2023-Q3,29,96.6,82.8,82.8,75.9,51.7,44.8
1,2023-Q4,45,86.7,82.2,75.6,73.3,60.0,53.3
2,2024-Q1,52,90.4,80.8,78.8,71.2,65.4,53.8
3,2024-Q2,70,85.7,77.1,71.4,61.4,50.0,44.3
4,2024-Q3,87,94.3,82.8,79.3,70.1,57.5,NaN
5,2024-Q4,81,90.1,80.2,74.1,67.9,59.3,NaN
6,2025-Q1,85,85.9,75.3,67.1,64.7,NaN,NaN
7,2025-Q2,92,87.0,77.2,68.5,59.8,NaN,NaN
8,2025-Q3,116,90.5,82.8,72.4,NaN,NaN,NaN
9,2025-Q4,100,88.0,84.0,3.0,NaN,NaN,NaN


### KPI 12 — Cohort Retention by Signup Quarter

**SQL techniques:** 2-CTE chain, `STRFTIME()` for quarter extraction,
`CASE WHEN` for quarter number, `JULIANDAY()` for checkpoint validation,
`NULLIF()` to replace future checkpoints with NULL, `GROUP BY`

| Cohort | Size | Month 3 | Month 6 | Month 9 | Month 12 | Month 18 | Month 24 |
|---|---|---|---|---|---|---|---|
| 2023-Q3 | 29 | 96.6% | 82.8% | 82.8% | 75.9% | 51.7% | 44.8% |
| 2023-Q4 | 45 | 86.7% | 82.2% | 75.6% | 73.3% | 60.0% | 53.3% |
| 2024-Q1 | 52 | 90.4% | 80.8% | 78.8% | 71.2% | 65.4% | 53.8% |
| 2024-Q2 | 70 | 85.7% | 77.1% | 71.4% | 61.4% | 50.0% | 44.3% |
| 2024-Q3 | 87 | 94.3% | 82.8% | 79.3% | 70.1% | 57.5% | — |
| 2024-Q4 | 81 | 90.1% | 80.2% | 74.1% | 67.9% | 59.3% | — |

**Key technique:**  
`NULLIF(value, 0)` converts zeros to NULL for checkpoints not yet reachable —  
preventing misleading 0% retention values for cohorts that simply haven't  
existed long enough. `STRFTIME()` with `CASE WHEN` extracts the quarter  
number from the month, a common SQLite pattern since it lacks `QUARTER()`.

**Key insight:**  
Retention drops most steeply between Month 3 and Month 9 across all cohorts.  
2024-Q2 is the weakest cohort (44.3% at 24 months) while 2023-Q4 and 2024-Q1  
are the strongest (53.3% and 53.8%). Month 9 survival is the critical threshold.

## 6. Key Findings — SQL Analysis

This notebook replicated all 12 KPIs from the Python EDA using SQL,
demonstrating that the same business insights can be derived through
multiple analytical tools — a critical skill in professional data analytics.

---

### SQL Techniques Demonstrated

| Technique | KPIs Used |
|---|---|
| Common Table Expressions (CTEs) | KPI 1, 2, 7, 8, 9, 10, 11, 12 |
| Window Function `SUM() OVER` | KPI 1 |
| Window Function `RANK() OVER` | KPI 8 |
| `JULIANDAY()` date arithmetic | KPI 6, 10, 11, 12 |
| `STRFTIME()` date formatting | KPI 12 |
| `CASE WHEN` conditional aggregation | KPI 2, 4, 5, 10, 11, 12 |
| `UNION ALL` inline reference table | KPI 1, 9 |
| Multi-table JOIN (up to 4 tables) | KPI 7, 8, 9 |
| `NULLIF()` for null handling | KPI 12 |
| Hybrid SQL + Python approach | KPI 3 |

---

### Consistency with Python Analysis

All KPIs produced results consistent with the Python EDA notebook,
confirming the integrity of both the dataset and the analytical logic.  
Minor differences in some KPIs (NRR, LTV by country) are attributable  
to methodological differences — documented inline where relevant.

---

### Key Business Findings (confirmed across both tools)

1. **MRR grew ~60x** over 36 months — strong acquisition-led growth
2. **NRR of 82%** — below 100% benchmark, churn exceeds expansion revenue
3. **Starter churn (52.1%)** is 5x higher than Enterprise (10.1%)
4. **Enterprise LTV (€10,769)** is 16x higher than Starter (€659)
5. **Early engagement predicts churn** — 11.4% fewer events in first 30 days
6. **Deep feature non-users** are 1.40x more likely to churn (48% vs 34%)
7. **Month 9** is the critical retention threshold across all cohorts

---

*SQL Analysis by Gabriel | Flowmetric SaaS Portfolio Project | July 2026*  
*Database: SQLite | Python + pandas for hybrid calculations*  
*Cross-validated against Python EDA notebook (flowmetric_eda.ipynb)*